# Diferentes métodos de recomendación:

### User-based:
recommendation = sum(s_u,v_ * r_v,i_) for v in V_u_
- siendo V_u_ el vecindario del usuario u, s la similaridad y r el rating de v para el item i 

==> Hay que ver cómo calcular el vecindario del usuario y la similaridad

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import load_npz, csr_matrix
import numpy as np
import zipfile
import json
import pickle

In [ ]:
input_dir = "../matrices/sparse_matrix_train.npz"

train_matrix = load_npz(input_dir)

In [ ]:
with zipfile.ZipFile("../datos/spotify_test_playlists.zip", "r") as zipf:
    for filename in zipf.namelist():
        if filename.endswith("test_input_playlists.json"):
            with zipf.open(filename) as f:
                test_data = json.loads(f.read())
            break

In [ ]:
with open("../mapas/tracks_train.pkl", "rb") as tr:
    track_to_idx_train = pickle.load(tr)
    
with open("../mapas/pid_train.pkl", "rb") as p:
    pid_to_idx_train = pickle.load(p)
    

In [ ]:
rows = []
cols = []

pid_to_idx_test = {}
track_to_idx_test = track_to_idx_train.copy()

current_pid_idx = 0
current_track_idx = len(track_to_idx_test)

for playlist in test_data["playlists"]:

    pid = playlist["pid"]

    # procesamos los pids de las playlist a índices
    if pid not in pid_to_idx_test: # realmente no haría falta, pero lo dejamos por si aca
        pid_to_idx_test[pid] = current_pid_idx
        current_pid_idx +=1

    p_idx = pid_to_idx_test[pid]

    # procesamos los tracks_uri a índices
    tracks = playlist.get("tracks", [])

    if tracks:
        for track in tracks:
            track_uri = track["track_uri"]

            if track_uri in track_to_idx_test:
                t_idx = track_to_idx_test[track_uri] # esto es diferente que antes porque en knn solo se pueden evaluar cosas vistas en train
            
                rows.append(p_idx)
                cols.append(t_idx)

num_playlist = len(pid_to_idx_test)
num_tracks = len(track_to_idx_test)

print(f"Número total de playlist: {num_playlist}")
print(f"Número total de tracks: {num_tracks}")

rows_np = np.array(rows, dtype = np.int32)
cols_np = np.array(cols, dtype = np.int32)
data_np = np.ones(len(rows_np), dtype = np.int8) # también podría ser la longitud de las columnas porque tenemos que colocar tantos 1s como coordenadas haya
                                                    # una coordenada está formada (valor_fila, valor_columna)

matrix = csr_matrix((data_np, (rows_np, cols_np)),
                    shape = (num_playlist, num_tracks),
                    dtype = np.int8)

In [ ]:
print(matrix.shape)
print(train_matrix.shape)

# primero voy a hacer user-based (playlist)

In [ ]:
from sklearn.preprocessing import normalize

# primero hay que normalizar los datos 
test_matrix_norm = normalize(matrix) # por defecto ya es normalización L2 por fila, por lo que no hace falta indicarlo
train_matrix_norm = normalize(train_matrix)

# configuración del batch (no lo hacemos todo de golpe) y resultados
k_vecinos = 100
batch_size = 100
num_test_playlist = test_matrix_norm.shape[0]

In [ ]:
print(f"Inciando el cálculo de vecinos de {num_test_playlist} playlists en batches de {batch_size}")

# para almacenar los resultados
indices_vecinos = np.zeros((num_test_playlist, k_vecinos), dtype=np.int32)
similitudes_coseno = np.zeros((num_test_playlist, k_vecinos), dtype=np.float32)

# bucle de procesamiento por lotes
for start_idx in range(0, num_test_playlist, batch_size):
    end_idx = min(start_idx+batch_size, num_test_playlist)

    # extraemos el lote actual
    batch_test = test_matrix_norm[start_idx:end_idx]

    # similitud coseno (producto escalar/dot product)
    similitud_batch = batch_test.dot(train_matrix_norm.T)

    # transformamos a array
    similitud_densa = similitud_batch.toarray()

    # Obtener los índices de los top-k usando partición
    partition_idx = np.argpartition(-similitud_densa, k_vecinos-1, axis=1)[:, :k_vecinos]
    top_k_vecinos = partition_idx[np.arange(len(partition_idx))[:, None], 
                                np.argsort(-similitud_densa[np.arange(len(similitud_densa))[:, None], 
                                            partition_idx], axis=1)]

    # nos quedamos con las similitudes de esos vecinos
    top_k_similitudes = np.take_along_axis(similitud_densa, top_k_vecinos, axis=1)

    # guardamos los resultados
    indices_vecinos[start_idx:end_idx] = top_k_vecinos
    similitudes_coseno[start_idx:end_idx] = top_k_similitudes

    print(f"Procesados los {k_vecinos} vecinos de {end_idx} playlists ({end_idx} de {num_test_playlist})")


In [ ]:
# Guardar como .npz (comprimido)
np.savez_compressed("resultados_knn.npz", 
                     indices=indices_vecinos,
                     similitudes=similitudes_coseno)

In [ ]:
# Cargar después
datos = np.load("resultados_knn.npz")
indices_vecinos = datos['indices']
similitudes_coseno = datos['similitudes']

In [ ]:
# 1. Contar cuántas filas son completamente cero
num_cero_rows = np.sum(np.all(similitudes_coseno == 0, axis=1))
print(f"Filas con similitud [0,0,0,0,0]: {num_cero_rows} de {num_test_playlist}")
print(f"Porcentaje: {num_cero_rows / num_test_playlist * 100:.1f}%")

# 2. Ver distribución de máximas similitudes
max_por_fila = similitudes_coseno.max(axis=1)
print(f"\nDistribución de max similitudes por fila:")
print(f"  = 0.0: {np.sum(max_por_fila == 0)}")
print(f"  0.0-0.1: {np.sum((max_por_fila > 0) & (max_por_fila <= 0.1))}")
print(f"  0.1-0.3: {np.sum((max_por_fila > 0.1) & (max_por_fila <= 0.3))}")
print(f"  > 0.3: {np.sum(max_por_fila > 0.3)}")

# 3. Verificar una playlist de test específica
test_playlist_idx = 0
print(f"\nPlaylist test {test_playlist_idx}:")
print(f"  Tracks: {matrix[test_playlist_idx].nnz}")  # nnz = elementos no-cero
print(f"  Vecinos encontrados: {indices_vecinos[test_playlist_idx,]}")
print(f"  Similitudes: {similitudes_coseno[test_playlist_idx,]}")

# 4. Verificar si hay tracks en común entre test y train
# test_tracks = set(matrix.nonzero()[1])
# train_tracks = set(train_matrix.nonzero()[1])
# tracks_en_comun = test_tracks & train_tracks
# print(f"\nTracks en test: {len(test_tracks)}")
# print(f"Tracks en train: {len(train_tracks)}")
# print(f"Tracks en común: {len(tracks_en_comun)}")
# print(f"Porcentaje overlaps: {len(tracks_en_comun) / len(test_tracks) * 100:.1f}%")

# obtenemos las recomendaciones en base al vecindario (user-based)

In [ ]:
# tengo guardado la similitud coseno y el índice de los vecinos
# cada playlist a la que le quiero recomendar algo, tiene k playlists vacías
# para cada playlist vecina, las tracks que contenga tendrán valor la similitud coseno de la playlist
# a cada track hay que asociarle un peso que va a ir incrementando a medida que aparezca en las playlist
# después, quedarme con aquellas tracks que tengan mayor peso (que probablemente sean las más populares)
idx_to_track_train = {v: k for k, v in track_to_idx_train.items()}
idx_to_pid_test = {v: k for k, v in pid_to_idx_test.items()}
results = {}

k = 10

for i in range(num_test_playlist):
    # para cada playlist a recomendar, obtengo las k vecinas
    # lo que tengo es el índice, por lo que tengo que sacarla de train_matrix
    pesos_tracks=np.zeros((1,train_matrix.shape[1]), np.float32)
    recommendations = []
    playlists_vecinas = train_matrix[indices_vecinos[i,:k].tolist()]
    # num_tracks_a_recomendar = 500 - matrix[i].nnz
    for v, similitud_coseno in enumerate(similitudes_coseno[i,:k]):
        # ahora voy a hacer la suma ponderada, al final tendré que pasar los idx de las tracks a su id
        # para obtener las mejores playlist lo que puedo hacer es inicializar un vector a 0 e ir sumándole 
        playlist_vecina = playlists_vecinas[v,].toarray().flatten()
        pesos_tracks[0] += similitud_coseno*playlist_vecina

    top_indices = np.argsort(pesos_tracks[0])[-1000:][::-1]

    for top_idx in top_indices:
        if matrix[i, top_idx]==0:
            track = idx_to_track_train[int(top_idx)]
            recommendations.append(track)
        if len(recommendations) == 500:
            break
    pid = idx_to_pid_test[i]
    results[pid] = recommendations
    print(f"Playlist {pid}: {len(recommendations)} recomendaciones")

In [ ]:
# !pip install tqdm

In [ ]:
idx_to_track_train = {v: k for k, v in track_to_idx_train.items()}
idx_to_pid_test = {v: k for k, v in pid_to_idx_test.items()}
results = {}
k=10

In [ ]:
from multiprocessing import Pool
from tqdm import tqdm

def procesar_playlist(i):
    pesos_tracks = np.zeros(train_matrix.shape[1], dtype=np.float32)
    playlists_vecinas = train_matrix[indices_vecinos[i, :k].tolist()]
    
    for v in range(k):
        pesos_tracks += similitudes_coseno[i, v] * playlists_vecinas[v, :].toarray().ravel()
    
    top_indices = np.argsort(-pesos_tracks)[:1000]
    mask = matrix[i, top_indices].toarray().ravel() == 0
    recommendations = [idx_to_track_train[int(idx)] for idx in top_indices[mask][:500]]
    
    pid = idx_to_pid_test[i]
    return pid, recommendations

# Procesar en paralelo con barra de progreso
with Pool(processes=8) as pool:
    results_list = list(tqdm(pool.imap(procesar_playlist, range(num_test_playlist)), 
                             total=num_test_playlist, 
                             desc="Procesando playlists"))
    results = {pid: recs for pid, recs in results_list}

print(f"Procesadas {len(results)} playlists")

In [ ]:
output_file_path = "resultado_baseline2.csv"

print(f"Guardando resultados en {output_file_path}...")
with open(output_file_path, "w") as f:
    f.write("team_info, Pablo Fernandez Rubal - Noura el Morchid - Gonzalo Ponte Rodriguez, pablo.fernandez.rubal@udc.es - n.elmorchid@udc.es - g.ponte@udc.es\n")
    for pid, tracks in results.items():
        linea = f"{pid}," + ",".join(tracks)
        f.write("\n" + linea + "\n")
        
print("Archivo generado correctamente")